# PostToolUse Hook Input Schema

**Purpose:** Reference for the PostToolUse hook input shape — one section per tool. Use this when writing hooks that need to inspect or react to tool invocations.

Source: `research/plugins/capture/` — self-driving capture plugin (runs `claude -p` to exercise tools and appends each PostToolUse event to a JSONL file)

## Background

Claude Code fires PostToolUse hooks **after** a tool completes. The hook receives a JSON payload on stdin with a shared envelope:

```json
{
  "session_id": "<uuid>",
  "tool_name": "<ToolName>",
  "tool_input": { ... },
  "tool_response": { ... }
}
```

- **`session_id`** — the active Claude Code session identifier (used to locate task files, log context, etc.)
- **`tool_name`** — which tool was just invoked (`Skill`, `Bash`, `Read`, etc.)
- **`tool_input`** — the arguments passed to the tool (varies per tool)
- **`tool_response`** — the tool's output (varies per tool)

PostToolUse hooks are configured in `hooks.json` and typically return `{"continue": true}` to allow execution to proceed.

In [ ]:
# Hook stdin reader — copy-paste boilerplate for any PostToolUse hook
import json
import sys

def read_hook_input() -> dict:
    """Read and parse the JSON payload from stdin (as Claude Code sends it)."""
    raw = sys.stdin.read()
    if not raw:
        return {}
    return json.loads(raw)

# In a real hook script:
# hook_data    = read_hook_input()
# session_id   = hook_data.get('session_id')
# tool_name    = hook_data.get('tool_name')
# tool_input   = hook_data.get('tool_input', {})
# tool_response = hook_data.get('tool_response', {})

print("Boilerplate ready — see comments above for usage in hook scripts.")

In [ ]:
# Capture PostToolUse events — clear prior run, invoke claude -p, load JSONL
import json
from pathlib import Path
from lib import run_claude

PROMPT = """Use each of these tools once with a minimal, safe operation:
1. Read: read /etc/hostname
2. Bash: echo hello
3. Glob: find *.md files in current directory
4. Grep: search for "def " in any Python file
Reply with "done" when finished."""

CAPTURE_FILE = Path("/tmp/claude/post-tool-use-capture.jsonl")
PLUGIN_DIR = Path("research/plugins/capture").resolve()
CAPTURE_FILE.unlink(missing_ok=True)

cr = run_claude(PROMPT, plugin_dir=PLUGIN_DIR)

print("Session ID:", cr.session_id)
print("Exit code:", cr.returncode)
if cr.stderr:
    print("stderr:", cr.stderr[:300])
print("JSON output keys:", list(cr.output.keys()))

# JSONL capture (hook envelopes) — unchanged
events = [json.loads(line) for line in CAPTURE_FILE.read_text().splitlines() if line]
print(f"\nCaptured {len(events)} PostToolUse event(s)")

In [ ]:
# Render per-tool envelope shapes from the captured events
from rich.pretty import pprint

by_tool = {}
for event in events:
    tool = event.get("tool_name", "unknown")
    by_tool.setdefault(tool, []).append(event)

for tool_name, tool_events in sorted(by_tool.items()):
    print(f"\n{'='*60}")
    print(f"  {tool_name}  ({len(tool_events)} event(s))")
    print(f"{'='*60}")
    sample = tool_events[0]
    print("\nEnvelope keys:", list(sample.keys()))
    print("\ntool_input:")
    pprint(sample.get("tool_input", {}))
    print("\ntool_response:")
    resp = sample.get("tool_response", {})
    resp_str = json.dumps(resp) if not isinstance(resp, str) else resp
    if len(resp_str) > 400:
        print(resp_str[:400] + " ... (truncated)")
    else:
        pprint(resp)

## Per-Tool Schemas (Reference)

Live-captured schemas are rendered by the cell above. The entries below are static reference docs; re-run the capture cell to refresh real data.

### `Read`

**`tool_input`:**
```json
{
  "file_path": "<absolute path to file>"
}
```

**`tool_response`:**
```json
{
  "content": "<file contents as a string>"
}
```

---

### `Bash`

**`tool_input`:**
```json
{
  "command": "<shell command string>",
  "description": "<human-readable description>"
}
```

**`tool_response`:**
```json
{
  "stdout": "<command stdout>",
  "stderr": "<command stderr>",
  "exit_code": 0
}
```

---

### `Glob`

**`tool_input`:**
```json
{
  "pattern": "<glob pattern>",
  "path": "<directory>"
}
```

**`tool_response`:** `{ "files": ["<path>", ...] }`

---

### `Grep`

**`tool_input`:**
```json
{
  "pattern": "<regex>",
  "path": "<directory>",
  "glob": "<file filter>",
  "output_mode": "files_with_matches | content | count"
}
```

**`tool_response`:** varies by `output_mode`

---

### Additional tools

To capture other tools (Write, Edit, Task, TodoWrite, WebFetch, Skill, etc.):

1. Edit `PROMPT` in the capture cell to exercise the target tool
2. Re-run the capture cell — the plugin appends a JSON envelope per PostToolUse event
3. Re-run the render cell to see the new schemas

## How It Works

The capture cell above is self-driving:

1. **Clears** `/tmp/claude/post-tool-use-capture.jsonl` from any prior run
2. **Runs** `claude -p` with `research/plugins/capture` loaded via `--plugin-dir`
3. **Captures** each PostToolUse event — the plugin's `capture.py` appends a JSON envelope per tool call
4. **Loads** the JSONL into `events` for the render cell to display

The capture plugin (`research/plugins/capture/`) requires no manual hook registration. It is loaded transiently via `--plugin-dir` and does not affect your normal Claude Code session.